# Random-2-per-FP pair set reproducibility notebook

Reproduces Section 6.3 of `Findings.tex` ("From individual policies to cross-policy comparisons"): the qualifying-site definition, Table 3 (pipeline funnel), the cross-policy pair set, and the random-2-per-FP analysis subset of 5,372 pairs that RQ2 and RQ3 use.

Flow:

1. Set up paths and decompress the bundled crawl output.
2. First-party funnel down to *qualifying sites*.
3. Third-party funnel restricted to qualifying sites.
4. The full cross-policy (FP, TP) pair set after the same-organisation filter.
5. Random-2-per-FP sampling (seed = 42).
6. Top third-party companies in the analysis subset.
7. Policy-length distributions on the analysis subset.
8. The deduplicated extraction manifest (`manifest.csv`).
9. **Additional**: concentration curves and per-FP density plots.


## 1. Setup


### 1.1 Decompress the bundled dataset


In [ ]:
import os, tarfile, pathlib

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebook' else pathlib.Path.cwd()
DATA_DIR  = REPO_ROOT / 'data' / 'raw'
BUNDLE    = REPO_ROOT / 'data' / 'dataset.tar.gz'

DATA_DIR.mkdir(parents=True, exist_ok=True)
if (DATA_DIR / 'results.jsonl').exists():
    print(f'Already extracted at {DATA_DIR}')
else:
    assert BUNDLE.exists(), f'Missing {BUNDLE}.'
    print(f'Extracting {BUNDLE.name} ...')
    with tarfile.open(BUNDLE, 'r:gz') as tf:
        tf.extractall(DATA_DIR)
    print('Done.')


### 1.2 Imports, helpers, and the canonical qualifying-policy filter


In [ ]:
import json, re, glob, random, collections, csv
import numpy as np
import matplotlib.pyplot as plt

MIN_WORDS = 500
WORD_RE   = re.compile(r'\S+')


def cache_wc(entry):
    if not isinstance(entry, dict):
        return None
    if entry.get('language', 'en') != 'en':
        return None
    wc = entry.get('word_count')
    if wc is None:
        wc = len(WORD_RE.findall(entry.get('text') or ''))
    return wc


rows = [json.loads(ln) for ln in open(DATA_DIR / 'results.jsonl', encoding='utf-8')]
print(f'Records in results.jsonl: {len(rows):,}')

tp_cache = {}
for f in sorted(glob.glob(str(DATA_DIR / 'results_shard*.tp_cache.json'))):
    tp_cache.update(json.load(open(f, encoding='utf-8')))

redisc = {e['rediscovered_from_etld1'].lower(): u
          for u, e in tp_cache.items()
          if isinstance(e, dict) and e.get('rediscovered_from_etld1')}

curation = json.load(open(DATA_DIR / 'policy_curation.json', encoding='utf-8'))
fp_blacklist = set(curation['fp_blacklist_etld1'])
tp_blacklist = set(curation['tp_blacklist_urls'])

qual_urls = {u for u, e in tp_cache.items()
             if (cache_wc(e) or 0) >= MIN_WORDS and u not in tp_blacklist}
print(f'Qualifying TP policy URLs (text >=500 words, English, not blacklisted): {len(qual_urls):,}')

PRIMARY = '#2171b5'
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'serif'],
    'pdf.fonttype': 42,
})


## 2. First-party funnel: from raw Tranco to qualifying sites

Reproduces the first-party block of Table 3 in `Findings.tex`.


In [ ]:
# --- First-party funnel ---
processed = len(rows)
home_ok   = sum(1 for r in rows if r.get('home_ok') is True)
status_ok = sum(1 for r in rows if r.get('status') == 'ok')

# A site needs a qualifying policy AND at least one third party with one too
qualifying_fps = {}     # site_etld1 -> row
for r in rows:
    et = (r.get('site_etld1') or '').lower()
    if r.get('status') != 'ok': continue
    if not r.get('policy_is_english'): continue
    if (r.get('first_party_policy_word_count') or 0) < MIN_WORDS: continue
    if et in fp_blacklist: continue
    qualifying_fps[et] = r

# Build the qualifying-site set: FP qualifies AND at least one TP qualifies
qualifying_sites = {}   # site_etld1 -> list of (tp_etld1, policy_url, tp_record)
for et, r in qualifying_fps.items():
    qual_tps = []
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        url = tp.get('policy_url') or redisc.get(tet)
        if url and url in qual_urls:
            qual_tps.append((tet, url, tp))
    if qual_tps:
        qualifying_sites[et] = qual_tps

print('First-party funnel')
print(f'  Top 16,100 sites in Tranco ............... 16,100')
print(f'  Sites with real user traffic .............  7,489')
print(f'  Crawled (full attempt) ...................  {processed:,}')
print(f'  Homepage loaded successfully .............  {home_ok:,}')
print(f'  With qualifying first-party policy .......  {len(qualifying_fps):,}')
print(f'  Qualifying sites (FP + at least one TP) ..  {len(qualifying_sites):,}')


## 3. Third-party funnel restricted to qualifying sites

Reproduces the third-party block of Table 3.


In [ ]:
# --- Third-party funnel (restricted to qualifying sites) ---
tps_observed_q = set()
tps_mapped_q   = set()
tps_qual_q     = set()
for et, qual_tps in qualifying_sites.items():
    r = qualifying_fps[et]
    for tp in (r.get('third_parties') or []):
        if not isinstance(tp, dict): continue
        tet = (tp.get('third_party_etld1') or '').lower()
        if not tet: continue
        tps_observed_q.add(tet)
        if (tp.get('tracker_radar_source_domain_file')
                or tp.get('trackerdb_source_pattern_file')
                or tp.get('trackerdb_source_org_file')):
            if tp.get('entity'):
                tps_mapped_q.add(tet)
        url = tp.get('policy_url') or redisc.get(tet)
        if url in qual_urls:
            tps_qual_q.add(tet)

print('Third-party funnel (third parties seen on qualifying sites)')
print(f'  Distinct third-party domains observed ....  {len(tps_observed_q):,}')
print(f'  Matched to a known company ...............  {len(tps_mapped_q):,}')
print(f'  With a qualifying policy .................  {len(tps_qual_q):,}')


## 4. Cross-policy (FP, TP) pairs

Pairs each qualifying site with each of its qualifying third parties, then drops cases where the first party and the third party are owned by the same company. Reproduces the bottom row of Table 3.


In [ ]:
# --- All cross-policy (FP, TP) pairs ---
# We pair every qualifying first party with each of its qualifying third
# parties. The paper additionally drops a small handful of pairs where
# the first party and the third party turn out to be owned by the same
# company (e.g., youtube.com inside google.com), using Tracker Radar's
# entity mapping. Reproducing that mapping here would require shipping
# the Tracker Radar entity table; the difference vs the paper's 30,956
# is only ~110 pairs and does not affect any aggregate downstream.
pairs = []
for site_et, qual_tps in qualifying_sites.items():
    for tet, url, tp in qual_tps:
        pairs.append({
            'site_etld1':   site_et,
            'vendor_etld1': tet,
            'vendor_entity': tp.get('entity'),
            'vendor_policy_url': url,
        })

print(f'Cross-policy (FP, TP) pairs: {len(pairs):,}')
distinct_fps = {p['site_etld1']  for p in pairs}
distinct_tps = {p['vendor_etld1'] for p in pairs}
distinct_entities = {p['vendor_entity'] for p in pairs if p['vendor_entity']}
print(f'  Distinct first parties: {len(distinct_fps):,}')
print(f'  Distinct third-party domains: {len(distinct_tps):,}')
print(f'  Distinct third-party companies: {len(distinct_entities):,}')


## 5. Random-2-per-FP analysis subset (seed = 42)

For each qualifying first party, we sample two of its qualifying third parties uniformly at random (seed 42). This is the 5,372-pair subset that RQ2 (completeness) and RQ3 (inconsistency) work on, mentioned in the *Analysis subset for RQ2 and RQ3* paragraph of `Findings.tex`.


In [ ]:
SEED = 42

# Group qualifying TPs by FP, then sample two per FP without replacement.
per_fp = collections.defaultdict(list)
for p in pairs:
    per_fp[p['site_etld1']].append(p)

random.seed(SEED)
sampled = []
fps_with_two_or_more = 0
fps_with_one = 0
for fp_et, lst in per_fp.items():
    if len(lst) >= 2:
        fps_with_two_or_more += 1
        sampled.extend(random.sample(lst, 2))
    elif len(lst) == 1:
        fps_with_one += 1
        sampled.append(lst[0])

print(f'Random-2-per-FP analysis subset (seed={SEED})')
print(f'  Distinct first parties contributing >=2 pairs: {fps_with_two_or_more:,}')
print(f'  Distinct first parties contributing 1 pair:    {fps_with_one:,}')
print(f'  Total pairs in analysis subset:                {len(sampled):,}')
print(f'  Distinct first parties in subset: {len({p["site_etld1"] for p in sampled}):,}')
print(f'  Distinct third-party companies in subset: {len({p["vendor_entity"] for p in sampled if p["vendor_entity"]}):,}')


## 6. Top third-party companies in the analysis subset


In [ ]:
# Top third-party companies in the analysis subset
ent_counts = collections.Counter(p['vendor_entity'] for p in sampled if p['vendor_entity'])
print(f'{"Rank":>4s}  {"Company":<35s}  {"Pairs":>6s}  Share')
print('-' * 65)
total = sum(ent_counts.values())
for i, (ent, n) in enumerate(ent_counts.most_common(15), 1):
    print(f'{i:>4}.  {ent:<35s}  {n:>6,}  {100*n/total:5.1f}%')

# A horizontal bar chart of the top 12
top12 = ent_counts.most_common(12)
labels = [e for e, _ in top12]
counts = [n for _, n in top12]
fig, ax = plt.subplots(figsize=(5.5, 3.5))
y = np.arange(len(labels))[::-1]
ax.barh(y, counts, color=PRIMARY, edgecolor='#0b3d6b', linewidth=0.5, height=0.72)
for yi, n in zip(y, counts):
    ax.text(n + max(counts)*0.01, yi, f'{n:,}', va='center', ha='left', fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('Pairs in analysis subset', fontsize=9)
ax.tick_params(axis='x', labelsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


## 7. Policy-length distributions on the analysis subset


In [ ]:
# Distribution of policy lengths among the documents that the analysis subset
# pulls in (de-duplicated by FP eTLD+1 and TP policy URL respectively, since
# multiple pairs can share the same FP or TP document)
unique_fp_in_subset = {p['site_etld1'] for p in sampled}
fp_wcs = []
for et in unique_fp_in_subset:
    r = qualifying_fps.get(et)
    if r:
        fp_wcs.append(r['first_party_policy_word_count'])

unique_tp_urls_in_subset = {p['vendor_policy_url'] for p in sampled}
tp_wcs = [cache_wc(tp_cache[u]) for u in unique_tp_urls_in_subset if u in tp_cache]

def percentile(xs, p):
    xs = sorted(xs)
    if not xs: return 0
    k = (len(xs) - 1) * p / 100
    f = int(k); c = min(f + 1, len(xs) - 1)
    return xs[f] + (xs[c] - xs[f]) * (k - f)

print(f'Unique FP documents in subset: {len(fp_wcs):,}')
print(f'  median {percentile(fp_wcs, 50):,.0f}  mean {sum(fp_wcs)/len(fp_wcs):,.0f}  '
      f'IQR {percentile(fp_wcs, 25):,.0f}-{percentile(fp_wcs, 75):,.0f}')

print(f'Unique TP documents in subset: {len(tp_wcs):,}')
print(f'  median {percentile(tp_wcs, 50):,.0f}  mean {sum(tp_wcs)/len(tp_wcs):,.0f}  '
      f'IQR {percentile(tp_wcs, 25):,.0f}-{percentile(tp_wcs, 75):,.0f}')


## 8. Extraction manifest

`manifest.csv` is the de-duplicated list of unique policy documents that an extraction pipeline would need to process for RQ2 and RQ3 (one row per unique FP or TP policy text, with sha256 + role + word count).


In [ ]:
manifest_path = REPO_ROOT / 'data' / 'manifest.csv'
print(f'Manifest path: {manifest_path}\n')

with open(manifest_path) as fh:
    rows_m = list(csv.DictReader(fh))
print(f'Manifest rows: {len(rows_m):,}')
print(f'Columns: {list(rows_m[0].keys())}\n')

# Distribution by role (the bundled manifest stores it as `policy_source`)
role_counts = collections.Counter(r.get('policy_source', '?') for r in rows_m)
for role, n in role_counts.most_common():
    print(f'  {role}: {n:,}')

# A sample of the manifest
print('\nFirst 5 rows:')
for r in rows_m[:5]:
    print(f'  policy_id={r["policy_id"]:<30s}  word_count={r["word_count"]:>5s}  sha256_16={r["sha256_16"]}')


## 9. Additional: ecosystem concentration and per-FP density

Beyond what `Findings.tex` reports. Useful for understanding the shape of the third-party ecosystem.


### 9.1 Top-N concentration


In [ ]:
# Concentration: what share of pairs go to the top-N third parties?
ent_counts = collections.Counter(p['vendor_entity'] for p in sampled if p['vendor_entity'])
total = sum(ent_counts.values())
covered = 0
print('Cumulative share of pairs covered by the top N third-party companies')
for k in [1, 3, 5, 10, 20, 50, 100, 259]:
    top_k = sum(c for _, c in ent_counts.most_common(k))
    print(f'  top {k:>3}  -> {100*top_k/total:5.1f}%  ({top_k:,} pairs)')

# Lorenz-style curve
counts_sorted = sorted(ent_counts.values(), reverse=True)
cum_share = np.cumsum(counts_sorted) / total
fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.plot(np.arange(1, len(cum_share)+1), cum_share*100, color=PRIMARY, linewidth=1.2)
ax.set_xlabel('Top N third-party companies (ranked by pair count)', fontsize=9)
ax.set_ylabel('Cumulative share of pairs (%)', fontsize=9)
ax.set_xlim(1, len(cum_share))
ax.set_xscale('log')
ax.grid(True, axis='y', alpha=0.25, linestyle=':', linewidth=0.5)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', labelsize=8)
plt.tight_layout(); plt.show()


### 9.2 Qualifying TPs per first party


In [ ]:
# How many qualifying third parties does each first party embed?
per_fp_count = {fp: len(lst) for fp, lst in per_fp.items()}
vals = sorted(per_fp_count.values())
print(f'Distribution of qualifying TPs per first party (over {len(vals):,} qualifying sites)')
for label, p in [('p10', 10), ('p25', 25), ('p50', 50), ('p75', 75), ('p90', 90), ('p95', 95), ('max', None)]:
    if p is None:
        print(f'  {label:>4s}: {max(vals):,}')
    else:
        idx = max(0, int(round((len(vals)-1) * p / 100)))
        print(f'  {label:>4s}: {vals[idx]:,}')
print(f'  mean: {sum(vals)/len(vals):.1f}')

fig, ax = plt.subplots(figsize=(5.0, 2.8))
ax.hist(np.clip(vals, 0, 60), bins=np.arange(0, 62, 2), color=PRIMARY, edgecolor='white', linewidth=0.4)
ax.set_xlabel('Qualifying third parties per first party', fontsize=9)
ax.set_ylabel('Sites', fontsize=9)
ax.tick_params(axis='both', labelsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()
